# LeBron James Points Prediction — Exploratory Data Analysis

This notebook explores the master dataset built by the data pipeline.
Run `python main.py fetch-data` and `python main.py build-features` before executing this notebook.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.alpha'] = 0.3

from src.config import get_config
cfg = get_config()
print('Config loaded. Player:', cfg.player_name)

In [ ]:
# Load feature dataset
features_path = cfg.processed_path / 'features.parquet'
if features_path.exists():
    df = pd.read_parquet(features_path)
    print(f'Loaded {len(df)} games, {df.shape[1]} features')
    print(df.dtypes.value_counts())
else:
    print('Run: python main.py build-features')

In [ ]:
# Points distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['PTS'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(df['PTS'].mean(), color='red', linestyle='--', label=f'Mean: {df["PTS"].mean():.1f}')
axes[0].set_title('LeBron Points Distribution')
axes[0].set_xlabel('Points')
axes[0].legend()

df.boxplot(column='PTS', by='SEASON', ax=axes[1], rot=45)
axes[1].set_title('Points by Season')
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Points')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Rolling 10-game average over career
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df['GAME_DATE'], df['PTS'], alpha=0.3, color='steelblue', label='Actual')
ax.plot(df['GAME_DATE'], df['PTS'].rolling(10).mean(), color='royalblue', linewidth=2, label='10-Game Rolling Avg')
ax.set_title('LeBron James — Points Scored Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Points')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Home vs Away scoring
if 'HOME_GAME' in df.columns:
    home_pts = df[df['HOME_GAME'] == 1]['PTS']
    away_pts = df[df['HOME_GAME'] == 0]['PTS']
    print(f'Home avg: {home_pts.mean():.1f}  Away avg: {away_pts.mean():.1f}')

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(home_pts, bins=30, alpha=0.6, label=f'Home ({home_pts.mean():.1f} avg)', color='steelblue', edgecolor='white')
    ax.hist(away_pts, bins=30, alpha=0.6, label=f'Away ({away_pts.mean():.1f} avg)', color='tomato', edgecolor='white')
    ax.set_title('Home vs Away Scoring Distribution')
    ax.set_xlabel('Points')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Back-to-back impact
if 'BACK_TO_BACK' in df.columns:
    b2b = df[df['BACK_TO_BACK'] == 1]['PTS']
    rested = df[df['BACK_TO_BACK'] == 0]['PTS']
    print(f'B2B avg: {b2b.mean():.1f}  Rested avg: {rested.mean():.1f}  Difference: {b2b.mean()-rested.mean():.1f}')

In [ ]:
# Correlation with Vegas implied team total
if 'IMPLIED_TEAM_TOTAL' in df.columns:
    valid = df[['PTS', 'IMPLIED_TEAM_TOTAL']].dropna()
    corr = valid.corr().loc['PTS', 'IMPLIED_TEAM_TOTAL']
    print(f'Correlation: LeBron PTS vs Implied Team Total = {corr:.3f}')

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(valid['IMPLIED_TEAM_TOTAL'], valid['PTS'], alpha=0.4, color='steelblue')
    ax.set_xlabel('Implied Team Total')
    ax.set_ylabel('Actual Points')
    ax.set_title(f'LeBron Points vs Implied Team Total (r={corr:.2f})')
    plt.tight_layout()
    plt.show()

In [ ]:
# Prospect Theory Index distribution
if 'PROSPECT_THEORY_INDEX' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    df['PROSPECT_THEORY_INDEX'].hist(bins=40, ax=axes[0], color='purple', edgecolor='white', alpha=0.7)
    axes[0].set_title('Prospect Theory Index Distribution')

    axes[1].scatter(df['PROSPECT_THEORY_INDEX'], df['PTS'], alpha=0.3, color='purple')
    axes[1].set_xlabel('Prospect Theory Index')
    axes[1].set_ylabel('Points')
    axes[1].set_title('PTS vs Prospect Theory Index')
    plt.tight_layout()
    plt.show()

In [ ]:
# Feature correlation with target
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'PTS' in numeric_cols:
    corr_with_pts = df[numeric_cols].corr()['PTS'].drop('PTS').sort_values(key=abs, ascending=False)
    top20 = corr_with_pts.head(20)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['steelblue' if v > 0 else 'tomato' for v in top20.values]
    ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1])
    ax.set_title('Top 20 Features Correlated with LeBron Points')
    ax.set_xlabel('Pearson Correlation')
    ax.axvline(0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# Missing value summary
missing = df.isnull().mean().sort_values(ascending=False)
missing_nonzero = missing[missing > 0]
print(f'Columns with missing values: {len(missing_nonzero)}')
if len(missing_nonzero) > 0:
    fig, ax = plt.subplots(figsize=(10, max(5, len(missing_nonzero[:30]) * 0.3)))
    missing_nonzero[:30].plot.barh(ax=ax, color='coral')
    ax.set_title('Missing Value Rate by Feature (top 30)')
    ax.set_xlabel('Fraction Missing')
    plt.tight_layout()
    plt.show()